[← Back to 01 - What Is a Database](<../01-what-is-a-database.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

# 01 - What Is a Database: Solutions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samietex/SQL_Tutorial/blob/main/solutions/01-what-is-a-database-solutions.ipynb)

Try each exercise in the lesson notebook first. If your query differs from the one here but returns the same rows, it may well be just as correct, and seeing why is a useful exercise in itself.

In [1]:
# Run this cell first. It downloads the small helper that lets the cells below run SQL,
# fetching a fresh copy every time so you always have the latest version.
import os, sys, tempfile, urllib.request
helper_dir = os.path.join(tempfile.gettempdir(), 'sql_tutorial_helper')
os.makedirs(helper_dir, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/samietex/SQL_Tutorial/assets/sql_tutorial.py',
    os.path.join(helper_dir, 'sql_tutorial.py'),
)
sys.path.insert(0, helper_dir)
%reload_ext sql_tutorial

Ready. %%sql runs SQLite 3.50.4 in memory, %%tsql runs SQL Server.


In [2]:
%%sql
CREATE TABLE signup_sheet (
    member_name TEXT,
    member_email TEXT,
    class_name TEXT,
    class_day TEXT
);
INSERT INTO signup_sheet VALUES
    ('Kemi', 'kemi@example.com', 'Spin', 'Monday'),
    ('Kemi', 'kemi@example.com', 'Yoga', 'Wednesday'),
    ('Tolu', 'tolu@example.com', 'Spin', 'Monday'),
    ('Musa', 'musa@example.com', 'Yoga', 'Wednesday');
SELECT
    member_name,
    member_email,
    class_name,
    class_day
FROM signup_sheet;

4 rows inserted


member_name,member_email,class_name,class_day
Kemi,kemi@example.com,Spin,Monday
Kemi,kemi@example.com,Yoga,Wednesday
Tolu,tolu@example.com,Spin,Monday
Musa,musa@example.com,Yoga,Wednesday


## 1. What does one row represent?

One row is **one member signed up for one class**, not one member and not one class. That's why Kemi appears twice: she has two sign-ups.

## 2. Which facts repeat, and how would you split them?

Kemi's email is written twice, and each class's day is written once for every member in it, so changing the Spin class to Tuesday would mean editing every Spin row, with the same risk of missing one that you saw with Ada's credit limit. A cleaner design has three tables:

- **members**: one row per member (an ID, a name, an email)
- **classes**: one row per class (an ID, a name, a day)
- **signups**: one row per member in a class, holding only a member ID and a class ID, the same way each order holds only a `customer_id`

Here it is as real tables. You don't need to read the code, just run it and look at the three results:

In [3]:
%%sql
CREATE TABLE members (
    member_id INTEGER PRIMARY KEY,
    member_name TEXT NOT NULL,
    member_email TEXT NOT NULL
);
CREATE TABLE classes (
    class_id INTEGER PRIMARY KEY,
    class_name TEXT NOT NULL,
    class_day TEXT NOT NULL
);
CREATE TABLE signups (
    member_id INTEGER NOT NULL,
    class_id INTEGER NOT NULL
);
INSERT INTO members VALUES
    (1, 'Kemi', 'kemi@example.com'),
    (2, 'Tolu', 'tolu@example.com'),
    (3, 'Musa', 'musa@example.com');
INSERT INTO classes VALUES
    (1, 'Spin', 'Monday'),
    (2, 'Yoga', 'Wednesday');
INSERT INTO signups VALUES
    (1, 1),
    (1, 2),
    (2, 1),
    (3, 2);

3 rows inserted
2 rows inserted
4 rows inserted


In [4]:
%%sql
SELECT
    member_id,
    member_name,
    member_email
FROM members;

member_id,member_name,member_email
1,Kemi,kemi@example.com
2,Tolu,tolu@example.com
3,Musa,musa@example.com


In [5]:
%%sql
SELECT
    class_id,
    class_name,
    class_day
FROM classes;

class_id,class_name,class_day
1,Spin,Monday
2,Yoga,Wednesday


In [6]:
%%sql
SELECT
    member_id,
    class_id
FROM signups;

member_id,class_id
1,1
1,2
2,1
3,2


Every fact now lives in one place. Moving Spin to Tuesday is a change to one row of `classes`, and the sign-up sheet can be put back together whenever someone wants to see it:

In [7]:
%%sql
UPDATE classes
SET class_day = 'Tuesday'
WHERE class_id = 1;
SELECT
    m.member_name,
    m.member_email,
    c.class_name,
    c.class_day
FROM signups AS s
INNER JOIN members AS m
    ON s.member_id = m.member_id
INNER JOIN classes AS c
    ON s.class_id = c.class_id
ORDER BY
    m.member_name,
    c.class_name;

1 row updated


member_name,member_email,class_name,class_day
Kemi,kemi@example.com,Spin,Tuesday
Kemi,kemi@example.com,Yoga,Wednesday
Musa,musa@example.com,Yoga,Wednesday
Tolu,tolu@example.com,Spin,Tuesday


Both Spin sign-ups show Tuesday after a one-row change.

**If you came up with two tables instead of three,** look at where the sign-ups would live. A member can take several classes and a class has several members, so if you put class columns on `members`, Kemi needs two rows again, and if you put member columns on `classes`, Spin needs two rows. Neither table can hold the link without repeating itself, which is why the link gets a table of its own. This shape, two tables connected through a third that holds only their IDs, is called a **many-to-many** relationship, and you'll see it constantly in real databases.

---

[← Back to 01 - What Is a Database](<../01-what-is-a-database.ipynb>) · [Course Overview](<../00-course-overview.ipynb>)

*SQL_Tutorial* is written and maintained by Samuel Shaibu as part of *All About Data & More*. Licensed under [MIT](../LICENSE).